# 지문자 단일 프레임 모델 학습
경로와 모델 버전만 수정하고, 학습 로직은 `fingerspellingAi.training`의 공통 구현을 사용한다.

In [ ]:
from pathlib import Path
import json
import sys
import torch

workingRoot = Path.cwd().resolve()
candidates = [workingRoot, workingRoot / "ai"]
for parent in workingRoot.parents:
    candidates.extend([parent, parent / "ai"])
aiRoot = next(path for path in candidates if (path / "src" / "fingerspellingAi").is_dir())
sys.path.insert(0, str(aiRoot / "src"))

print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
datasetRoot = Path("/data/Signlanguage-processed-v1")
outputRoot = Path("/data/artifacts/fingerspelling-v1.0.0")
modelVersion = "fingerspelling-v1.0.0"
gitCommit = "REPLACE_WITH_GIT_COMMIT_SHA"
evaluateTest = False
baselinePackage = None

In [ ]:
from fingerspellingAi.trainingData import loadTrainingDataset

dataset = loadTrainingDataset(
    datasetRoot,
    aiRoot / "config" / "labels.json",
    aiRoot / "config" / "preprocessing.json",
)
{
    "shape": dataset.features.shape,
    "train": len(dataset.indicesForSplit("train")),
    "validation": len(dataset.indicesForSplit("validation")),
    "test": len(dataset.indicesForSplit("test")),
}

In [ ]:
from fingerspellingAi.training import runTraining

def showProgress(epoch, total, metrics):
    if epoch == 1 or epoch % 5 == 0:
        print(epoch, metrics)

manifest = runTraining(
    datasetRoot=datasetRoot,
    outputRoot=outputRoot,
    modelVersion=modelVersion,
    labelsPath=aiRoot / "config" / "labels.json",
    preprocessingConfigPath=aiRoot / "config" / "preprocessing.json",
    recognitionPolicyPath=aiRoot / "config" / "recognition-policy.json",
    trainingConfigPath=aiRoot / "config" / "training.json",
    gitCommit=gitCommit,
    evaluateTest=evaluateTest,
    baselinePackage=baselinePackage,
    progressCallback=showProgress,
)
manifest

In [ ]:
metrics = json.loads((outputRoot / "metrics.json").read_text(encoding="utf-8"))
{
    "validation": {key: metrics["validation"][key] for key in ("accuracy", "macroF1", "expectedCalibrationError")},
    "testEvaluated": metrics["testEvaluated"],
    "report": str(outputRoot / "training-report.html"),
}